In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Standard imports
import numpy as np
import xarray as xr
import tqdm as tqdm

# For variogram estimation
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import label
from scipy.spatial.distance import pdist
from scipy.optimize import curve_fit

In [3]:
# Import OpenSense modules as submodules
import sys
import os

sys.path.append(os.path.abspath("./pycomlink/"))
sys.path.append(os.path.abspath("./poligrain/src/"))
sys.path.append(os.path.abspath("./mergeplg/src/"))

import pycomlink as pycml 
import poligrain as plg
import mergeplg 

In [4]:
os.makedirs('data/adjusted_fields', exist_ok=True)

In [5]:
# Define function to estimate variogram from rainfall event
def get_event_variogram(da_event, bin_edges, min_obs, plot_variogram=False):
    """
    da_event: xarray data for computing variogram
    bin_edges: array of distance bins [m] (e.g., np.linspace(0, 30000, 1500))
    min_obs: minimum observations needed to perform variogram estimation
    plot_variogram: whether to plot the variogram
    """
    all_distances = []
    all_sq_diffs = []
    
    # 1. Collect pairs from all time steps in the event
    n_obs = 0
    for t in da_event.time:
        # Extract data for this timestamp and drop nan
        data_t = da_event.sel(time=t).dropna(dim='cml_id')
        
        # We need at least 2 points to make a pair
        if len(data_t.cml_id) < 2:
            continue

        n_obs += data_t.cml_id.size

        # Get coordinates of data
        coords = np.column_stack([data_t.x, data_t.y])
        values = data_t.values
        
        # Calculate distances and 0.5 * (zi - zj)^2
        dist = pdist(coords, metric='euclidean')

        # Calculate squared distance (variance)
        sq_diff = 0.5 * pdist(values[:, None], 'sqeuclidean')
        
        all_distances.append(dist)
        all_sq_diffs.append(sq_diff)

    # If not enough observations
    if n_obs < min_obs:
        return None
        
    # Flatten into two long arrays of all pairs in the event
    dist_pool = np.concatenate(all_distances)
    diff_pool = np.concatenate(all_sq_diffs)
    
    # 2. Binning 
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    gamma_obs = []
    count = []

    # Also drop largest outliers
    #diff_upper = np.nanquantile(dist_pool, q = 0.95)
    
    for i in range(len(bin_edges)-1):
        mask = (dist_pool > bin_edges[i]) & (dist_pool <= bin_edges[i+1]) # & (diff_pool <= diff_upper) 
        if np.any(mask):
            gamma_obs.append(np.mean(diff_pool[mask]))
            count.append(diff_pool[mask].size)
        else:
            gamma_obs.append(np.nan)
            count.append(0)

    count = np.array(count)
    gamma_obs = np.array(gamma_obs)

    # 3. Define valid observation bins and get variance
    valid = ~np.isnan(gamma_obs) & (count != 0) 

    # Locate where most observations are
    dist_lower = np.nanquantile(dist_pool, q = 0.05)
    dist_upper = np.nanquantile(dist_pool, q = 0.95)
    ind_lower = np.where(dist_lower < bin_centers[valid])[0][0]

    if (dist_upper > bin_centers[valid]).all():
        ind_upper = bin_centers[valid].size
    else:
        ind_upper = np.where(dist_upper < bin_centers[valid])[0][0]

    total_variance = np.nanmean(gamma_obs[valid][ind_lower:ind_upper])
    gamma_obs_norm = gamma_obs/total_variance

    # If no variance 
    if total_variance == 0: 
        return None
        
    # 4. Define variogram and bounds
    def spherical_model(h, nugget, p_sill, range_a):
        # Use same definitions as pykrige:
        # https://geostat-framework.readthedocs.io/projects/pykrige/en/stable/variogram_models.html
        return np.where(h <= range_a, 
                        p_sill * (1.5 * (h/range_a) - 0.5 * (h/range_a)**3) + nugget, 
                        p_sill + nugget)
        
    # Estimate partial sill from normalized variance
    def f(h, n, r):
        return spherical_model(h, n, 1 - n, r)
        
    # Initial guess: [min(gamma), max_dist/2]
    p0 = [0, np.max(bin_centers[valid][ind_lower:ind_upper])/2]

    # Parameter bounds
    bound_l_nugget = 0
    bound_l_range = bin_centers[valid][ind_lower]
    bound_u_nugget = np.min([np.nanmean(gamma_obs_norm[valid][:ind_lower+1]), 1])
    bound_u_range = np.max(bin_edges)*2

    if bound_u_nugget == 0:
        bound_u_nugget = 0.01 # For numerics
    
    bounds = [[bound_l_nugget, bound_l_range], [bound_u_nugget, bound_u_range]]
    
    # 5. Optimize and return parameters
    popt, _ = curve_fit(
        f, 
        bin_centers[valid][:ind_upper], 
        gamma_obs_norm[valid][:ind_upper], 
        p0=p0, 
        bounds=bounds,
    )

    nugget = popt[0]
    p_sill = 1 - nugget
    range_a = popt[1]

    if plot_variogram:
        fig, ax = plt.subplots(1, 1)
        ax.plot(bin_centers[valid], spherical_model(bin_centers[valid], nugget, p_sill, range_a), label='plot variogram')
        ax.plot(bin_centers[valid][:ind_upper], gamma_obs_norm[valid][:ind_upper], label='data used')
        ax.plot(bin_centers[valid][ind_upper:], gamma_obs_norm[valid][ind_upper:], label='data left out')
        plt.legend()
        plt.show()
    
    return n_obs, [nugget, p_sill, range_a]


In [6]:
# Define function to estimate variogram from rainfall event
def get_event_variogram_fixnugget(da_event, bin_edges, min_obs, nugget, plot_variogram=False):
    """
    da_event: xarray data for computing variogram
    bin_edges: array of distance bins [m] (e.g., np.linspace(0, 30000, 1500))
    min_obs: minimum observations needed to perform variogram estimation
    plot_variogram: whether to plot the variogram
    """
    all_distances = []
    all_sq_diffs = []
    
    # 1. Collect pairs from all time steps in the event
    n_obs = 0
    for t in da_event.time:
        # Extract data for this timestamp and drop nan
        data_t = da_event.sel(time=t).dropna(dim='cml_id')
        
        # We need at least 2 points to make a pair
        if len(data_t.cml_id) < 2:
            continue

        n_obs += data_t.cml_id.size

        # Get coordinates of data
        coords = np.column_stack([data_t.x, data_t.y])
        values = data_t.values
        
        # Calculate distances and 0.5 * (zi - zj)^2
        dist = pdist(coords, metric='euclidean')

        # Calculate squared distance (variance)
        sq_diff = 0.5 * pdist(values[:, None], 'sqeuclidean')
        
        all_distances.append(dist)
        all_sq_diffs.append(sq_diff)

    # If not enough observations
    if n_obs < min_obs:
        return None
        
    # Flatten into two long arrays of all pairs in the event
    dist_pool = np.concatenate(all_distances)
    diff_pool = np.concatenate(all_sq_diffs)
    
    # 2. Binning 
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    gamma_obs = []
    count = []

    # Also drop largest outliers
    #diff_upper = np.nanquantile(dist_pool, q = 0.95)
    
    for i in range(len(bin_edges)-1):
        mask = (dist_pool > bin_edges[i]) & (dist_pool <= bin_edges[i+1]) # & (diff_pool <= diff_upper) 
        if np.any(mask):
            gamma_obs.append(np.mean(diff_pool[mask]))
            count.append(diff_pool[mask].size)
        else:
            gamma_obs.append(np.nan)
            count.append(0)

    count = np.array(count)
    gamma_obs = np.array(gamma_obs)

    # 3. Define valid observation bins and get variance
    valid = ~np.isnan(gamma_obs) & (count != 0) 

    # Locate where most observations are
    dist_lower = np.nanquantile(dist_pool, q = 0.05)
    dist_upper = np.nanquantile(dist_pool, q = 0.95)
    ind_lower = np.where(dist_lower < bin_centers[valid])[0][0]

    if (dist_upper > bin_centers[valid]).all():
        ind_upper = bin_centers[valid].size
    else:
        ind_upper = np.where(dist_upper < bin_centers[valid])[0][0]

    total_variance = np.nanmean(gamma_obs[valid][ind_lower:ind_upper])

    # If no variance 
    if total_variance == 0: 
        return None

    gamma_obs_norm = gamma_obs/total_variance

    # 4. Define variogram and bounds
    def spherical_model(h, nugget, p_sill, range_a):
        # Use same definitions as pykrige:
        # https://geostat-framework.readthedocs.io/projects/pykrige/en/stable/variogram_models.html
        return np.where(h <= range_a, 
                        p_sill * (1.5 * (h/range_a) - 0.5 * (h/range_a)**3) + nugget, 
                        p_sill + nugget)
        
    # Estimate partial sill from normalized variance
    def f(h, r):
        return spherical_model(h, nugget, 1-nugget, r)
        
    # Initial guess: [min(gamma), max_dist/2]
    p0 = [np.max(bin_centers[valid][ind_lower:ind_upper])/2]

    # Parameter bounds
    bound_l_range = bin_centers[valid][ind_lower]
    bound_u_range = np.max(bin_edges)*2
    
    bounds = [bound_l_range, bound_u_range]
    
    # 5. Optimize and return parameters
    popt, _ = curve_fit(
        f, 
        bin_centers[valid][:ind_upper], 
        gamma_obs_norm[valid][:ind_upper], 
        p0=p0, 
        bounds=bounds,
    )

    nugget = nugget
    p_sill = 1 - nugget
    range_a = popt[0]

    if plot_variogram:
        fig, ax = plt.subplots(1, 1)
        ax.plot(bin_centers[valid], spherical_model(bin_centers[valid], nugget, p_sill, range_a), label='plot variogram')
        ax.plot(bin_centers[valid][:ind_upper], gamma_obs_norm[valid][:ind_upper], label='data used')
        ax.plot(bin_centers[valid][ind_upper:], gamma_obs_norm[valid][ind_upper:], label='data left out')
        plt.legend()
        plt.show()
    
    return n_obs, [nugget, p_sill, range_a]


In [7]:
def estimate_several_variograms(ds_cmls, target_variable):
    # Variogram difference radar-cml
    bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
    min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search
    
    # Group nearby rainfall (closer than 6 hours) to estimate variogram
    mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
    mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
    labels_raw, num_features = label(mask)
    labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)
    
    variograms = []
    # Estimate variogram for rainfall events
    for i in tqdm.tqdm(range(1, num_features + 1)):
        # neighbouring events to include
        n_neighbors = 0 
    
        # Event for variogram estimation
        target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
        combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
        # Expand neighbourhood until we have enough data
        expand = True
        while expand:
            # Get start-end time
            time_start = combined_event_time.time.values[0]
            time_end = combined_event_time.time.values[-1]
            time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
            
            # Estimate variogram 
            out = get_event_variogram( 
                ds_cmls[target_variable].sel(time = slice(time_start, time_end)), # 
                bin_edges, 
                min_obs,
                plot_variogram = False,
            )
    
            # If variogram estimation was successful, stop expanding
            if out is not None:
                expand = False 
                n_obs = out[0]
                variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
                
            else:
                # Timesteps included in event
                n_time = combined_event_time.time.size
    
                # Expand neighborhood
                n_neighbors +=1
                target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
                combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
        
                # Break loop if exand did not result in more timesteps
                if n_time >= combined_event_time.time.size:
                    expand = False
                    print('Warning: Not enough rainfall obs in time series, using standard instead')
                    variograms = [[time_mid, 0.2, 0.8, 30000]]
    
    variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'psill', 'range']).set_index('time')
    
    # Aggregate variogram parameters across timesteps
    variograms = variograms.resample('14D', label='left').mean()
    
    # Add first and last timestep of CML to variogram dataframe, completing the time series
    start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
    end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
    variograms = pd.concat([start_row, variograms, end_row]).sort_index()
    return variograms

In [8]:
def estimate_several_variograms_fixnugget(ds_cmls, target_variable, nugget):
    # Variogram difference radar-cml
    bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
    min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search
    
    # Group nearby rainfall (closer than 6 hours) to estimate variogram
    mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
    mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
    labels_raw, num_features = label(mask)
    labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)
    
    variograms = []
    # Estimate variogram for rainfall events
    for i in tqdm.tqdm(range(1, num_features + 1)):
        # neighbouring events to include
        n_neighbors = 0 
    
        # Event for variogram estimation
        target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
        combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
        # Expand neighbourhood until we have enough data
        expand = True
        while expand:
            # Get start-end time
            time_start = combined_event_time.time.values[0]
            time_end = combined_event_time.time.values[-1]
            time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
            
            # Estimate variogram 
            out = get_event_variogram_fixnugget( 
                ds_cmls[target_variable].sel(time = slice(time_start, time_end)), 
                bin_edges, 
                min_obs,
                nugget,
                plot_variogram = False,
            )
    
            # If variogram estimation was successful, stop expanding
            if out is not None:
                expand = False 
                n_obs = out[0]
                variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
                
            else:
                # Timesteps included in event
                n_time = combined_event_time.time.size
    
                # Expand neighborhood
                n_neighbors +=1
                target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
                combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
        
                # Break loop if exand did not result in more timesteps
                if n_time >= combined_event_time.time.size:
                    expand = False
                    print('Warning: Not enough rainfall obs in time series, using standard instead')
                    variograms = [[time_mid, 0.2, 0.8, 30000]]
    
    variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'psill', 'range']).set_index('time')
    
    # Aggregate variogram parameters across timesteps
    variograms = variograms.resample('14D', label='left').mean()
    
    # Add first and last timestep of CML to variogram dataframe, completing the time series
    start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
    end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
    variograms = pd.concat([start_row, variograms, end_row]).sort_index()
    return variograms

In [9]:
# Evaluation fucntion
def compute_metrics(field, field_name, dataset):
    # RAINFALL FIELDS AT THE RAIN GAUGES
    get_grid_at_points = plg.spatial.GridAtPoints(
        da_gridded_data=ds_rad.isel(time = 0), 
        da_point_data=ds_gauges.isel(time = 0),
        nnear=1,
        stat="best" 
    )

    ds_gauges[field_name] = get_grid_at_points(
        da_gridded_data=field,
        da_point_data=ds_gauges.rainfall_amount,  
    )

    # COMPUTE METRICS
    threshold = 0.2

    metric = pd.DataFrame([plg.validation.calculate_rainfall_metrics(
        reference=ds_gauges.rainfall_amount.values.flatten(),
        estimate=ds_gauges[field_name].values.flatten(),
        ref_thresh=threshold,
        est_thresh=threshold,
    )]) 
    metric['dataset'] = dataset
    metric['method'] = field_name

    metric.to_csv(field_name)

# OpenMRG adjustment

In [10]:
# OpenMRG
ds_rad = xr.open_dataset("data/andersson_2022_OpenMRG/radar/openmrg_rad.nc")                    
ds_cmls = xr.open_dataset("data/processed_cml_OpenMRG.nc")       
ds_gauges = xr.open_dataset('data/andersson_2022_OpenMRG/gauges/openmrg_gauges.nc')     

# Get radar along CML, used for variogram estimation
da_intersect_weights = plg.spatial.calc_sparse_intersect_weights_for_several_cmls(
    x1_line=ds_cmls.site_0_lon.values,
    y1_line=ds_cmls.site_0_lat.values,
    x2_line=ds_cmls.site_1_lon.values,
    y2_line=ds_cmls.site_1_lat.values,
    cml_id=ds_cmls.cml_id.values,
    x_grid=ds_rad.lon.values,
    y_grid=ds_rad.lat.values,
    grid_point_location='center',
)
ds_cmls['radar_along_cml'] = plg.spatial.get_grid_time_series_at_intersections(
    grid_data=ds_rad.rainfall_amount,
    intersect_weights=da_intersect_weights,
)

# Difference used for additive
ds_cmls['rainfall_difference'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc - ds_cmls.radar_along_cml, 
    np.nan
)

# Ratio used for multiplicative
ds_cmls['rainfall_ratio'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc/ds_cmls.radar_along_cml, 
    np.nan
)

# CML obs used for KED
ds_cmls['rainfall_cml'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc, 
    np.nan
)

# For IDW loop
months = ['2015-06', '2015-07', '2015-08']                           

In [11]:
# methods parameter : default version
nnear = 70 
diff_check_sel = 10
ratio_check_sel = (0.1, 15)

In [12]:
# additive POINT ORDINARY KRIGING vandebeek

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_add_p_ok_vandebeek.csv', 
    'OpenMRG'
)
  
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:24<00:00,  6.74it/s]


In [13]:
# additive POINT ORDINARY KRIGING variogram

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_add_p_ok_variogram.csv', 
    'OpenMRG'
)   
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:26<00:00,  6.28it/s]


In [14]:
# additive POINT ORDINARY KRIGING variogram_fixnugget0

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_add_p_ok_variogram_fixnugget0.csv', 
    'OpenMRG'
)      
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:22<00:00,  7.57it/s]


In [15]:
# additive POINT ORDINARY KRIGING variogram_fixnugget03

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.3)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_add_p_ok_variogram_fixnugget03.csv', 
    'OpenMRG'
)    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:25<00:00,  6.68it/s]


In [16]:
# additive POINT ORDINARY KRIGING variogram_fixnugget06

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.6)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_add_p_ok_variogram_fixnugget06.csv', 
    'OpenMRG'
)    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:26<00:00,  6.39it/s]


In [17]:
# multiplicative POINT ORDINARY KRIGING vandebeek
variograms = estimate_several_variograms(ds_cmls, 'rainfall_ratio')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)
 
compute_metrics(
    data,
    'data/metrics/OpenMRG_mul_p_ok_vandebeek.csv', 
    'OpenMRG'
)     
del data, merger

 90%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊               | 26/29 [00:11<00:01,  2.92it/s]/tmp/ipykernel_1871958/4072047553.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:18<00:00,  9.10it/s]


In [18]:
# multiplicative POINT ORDINARY KRIGING variogram

variograms = estimate_several_variograms(ds_cmls, 'rainfall_ratio')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_mul_p_ok_variogram.csv', 
    'OpenMRG'
)   
del data, merger

 90%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊               | 26/29 [00:11<00:01,  2.92it/s]/tmp/ipykernel_1871958/4072047553.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:19<00:00,  8.72it/s]


In [19]:
# multiplicative POINT ORDINARY KRIGING variogram_fixnugget0

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_ratio', 0)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_mul_p_ok_variogram_fixnugget0.csv', 
    'OpenMRG'
)  
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:19<00:00,  8.45it/s]


In [20]:
# multiplicative POINT ORDINARY KRIGING variogram_fixnugget03

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_ratio', 0.3)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_mul_p_ok_variogram_fixnugget03.csv', 
    'OpenMRG'
)   
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:19<00:00,  8.70it/s]


In [21]:
# multiplicative POINT ORDINARY KRIGING variogram_fixnugget06

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_ratio', 0.6)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_mul_p_ok_variogram_fixnugget06.csv', 
    'OpenMRG'
)    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:19<00:00,  8.79it/s]


In [22]:
# additive BLOCK ORDINARY KRIGING vandebeek

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_add_b_ok_vandebeek.csv', 
    'OpenMRG'
)   
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:24<00:00,  6.91it/s]


In [23]:
# additive BLOCK ORDINARY KRIGING variogram

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_add_b_ok_variogram.csv', 
    'OpenMRG'
) 
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:24<00:00,  6.80it/s]


In [12]:
# additive BLOCK ORDINARY KRIGING variogram_fixnugget0

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_add_b_ok_variogram_fixnugget0.csv', 
    'OpenMRG'
)   
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:24<00:00,  6.74it/s]


In [13]:
# additive BLOCK ORDINARY KRIGING variogram_fixnugget03

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.3)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_add_b_ok_variogram_fixnugget03.csv', 
    'OpenMRG'
)   
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:24<00:00,  6.91it/s]


In [14]:
# additive BLOCK ORDINARY KRIGING variogram_fixnugget06

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.6)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_add_b_ok_variogram_fixnugget06.csv', 
    'OpenMRG'
)    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:24<00:00,  6.75it/s]


In [15]:
# multiplicative BLOCK ORDINARY KRIGING vandebeek

variograms = estimate_several_variograms(ds_cmls, 'rainfall_ratio')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_mul_b_ok_vandebeek.csv', 
    'OpenMRG'
)   
del data, merger

 90%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊               | 26/29 [00:11<00:00,  3.07it/s]/tmp/ipykernel_1895472/4072047553.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:19<00:00,  8.49it/s]


In [16]:
# multiplicative BLOCK ORDINARY KRIGING variogram

variograms = estimate_several_variograms(ds_cmls, 'rainfall_ratio')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_mul_b_ok_variogram.csv', 
    'OpenMRG'
)      
del data, merger

 90%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊               | 26/29 [00:11<00:01,  2.90it/s]/tmp/ipykernel_1895472/4072047553.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:20<00:00,  8.38it/s]


In [17]:
# multiplicative BLOCK ORDINARY KRIGING variogram_fixnugget0

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_ratio', 0)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_mul_b_ok_variogram_fixnugget0.csv', 
    'OpenMRG'
)     
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:21<00:00,  7.68it/s]


In [18]:
# multiplicative BLOCK ORDINARY KRIGING variogram_fixnugget03

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_ratio', 0.3)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_mul_b_ok_variogram_fixnugget03.csv', 
    'OpenMRG'
)     
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:19<00:00,  8.54it/s]


In [19]:
# multiplicative BLOCK ORDINARY KRIGING variogram_fixnugget06

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_ratio', 0.6)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_mul_b_ok_variogram_fixnugget06.csv', 
    'OpenMRG'
)   
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:18<00:00,  8.97it/s]


In [20]:
# KED point vandebeek

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_ked_p_vandebeek.csv', 
    'OpenMRG'
)   
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:24<00:00,  6.79it/s]


In [21]:
# KED point variogram

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_ked_p_variogram.csv', 
    'OpenMRG'
)      
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:27<00:00,  6.15it/s]


In [22]:
# KED point variogram_fixnugget0

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_ked_p_variogram_fixnugget0.csv', 
    'OpenMRG'
)     
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:24<00:00,  6.84it/s]


In [23]:
# KED point variogram_fixnugget03

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.3)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_ked_p_variogram_fixnugget03.csv', 
    'OpenMRG'
)    
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:23<00:00,  7.23it/s]


In [24]:
# KED point variogram_fixnugget06

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.6)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_ked_p_variogram_fixnugget06.csv', 
    'OpenMRG'
)   
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:22<00:00,  7.38it/s]


In [25]:
# KED block vandebeek 

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_ked_b_vandebeek.csv', 
    'OpenMRG'
)   
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:23<00:00,  7.06it/s]


In [26]:
# KED block variogram

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_ked_b_variogram.csv', 
    'OpenMRG'
)   
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:24<00:00,  6.97it/s]


In [27]:
# KED block variogram_fixnugget0

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_ked_b_variogram_fixnugget0.csv', 
    'OpenMRG'
)       
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:24<00:00,  6.74it/s]


In [28]:
# KED block variogram_fixnugget03

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.3)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_ked_b_variogram_fixnugget03.csv', 
    'OpenMRG'
)       
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:22<00:00,  7.55it/s]


In [29]:
# KED block variogram_fixnugget06

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.6)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenMRG_ked_b_variogram_fixnugget06.csv', 
    'OpenMRG'
)   
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:25<00:00,  6.61it/s]


# OpenRainER adjustment

In [30]:
# OpenRainER
ds_rad = xr.open_dataset("data/covi_2024_OpenRainER/openrainer_radar.nc")         
ds_cmls = xr.open_dataset("data/processed_cml_OpenRainER.nc")   
ds_gauges = xr.open_dataset('data/covi_2024_OpenRainER/AWS_rainfall.nc')        

# Get radar along CML, used for variogram estimation
da_intersect_weights = plg.spatial.calc_sparse_intersect_weights_for_several_cmls(
    x1_line=ds_cmls.site_0_lon.values,
    y1_line=ds_cmls.site_0_lat.values,
    x2_line=ds_cmls.site_1_lon.values,
    y2_line=ds_cmls.site_1_lat.values,
    cml_id=ds_cmls.cml_id.values,
    x_grid=ds_rad.lon.values,
    y_grid=ds_rad.lat.values,
    grid_point_location='center',
)
ds_cmls['radar_along_cml'] = plg.spatial.get_grid_time_series_at_intersections(
    grid_data=ds_rad.rainfall_amount,
    intersect_weights=da_intersect_weights,
)

# Difference used for additive
ds_cmls['rainfall_difference'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc - ds_cmls.radar_along_cml, 
    np.nan
)

# Ratio used for multiplicative
ds_cmls['rainfall_ratio'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc/ds_cmls.radar_along_cml, 
    np.nan
)

# CML obs used for KED
ds_cmls['rainfall_cml'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc, 
    np.nan
)   

months = ['2022-06', '2022-07', '2022-08']

In [31]:
# methods parameter : default version
nnear = 70 
diff_check_sel = 10
ratio_check_sel = (0.1, 15)

In [32]:
# additive POINT ORDINARY KRIGING vandebeek

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_add_p_ok_vandebeek.csv', 
    'OpenRainER'
)   
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [03:07<00:00,  1.41it/s]


In [33]:
# additive POINT ORDINARY KRIGING variogram

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_add_p_ok_variogram.csv', 
    'OpenRainER'
)       
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:35<00:00,  1.70it/s]


In [34]:
# additive POINT ORDINARY KRIGING variogram_fixnugget0

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_add_p_ok_variogram_fixnugget0.csv', 
    'OpenRainER'
)     
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:39<00:00,  1.65it/s]


In [35]:
# additive POINT ORDINARY KRIGING variogram_fixnugget03

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.3)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_add_p_ok_variogram_fixnugget03.csv', 
    'OpenRainER'
)   
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:40<00:00,  1.65it/s]


In [36]:
# additive POINT ORDINARY KRIGING variogram_fixnugget06

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.6)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_add_p_ok_variogram_fixnugget06.csv', 
    'OpenRainER'
)    
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:39<00:00,  1.65it/s]


In [37]:
# multiplicative POINT ORDINARY KRIGING vandebeek

variograms = estimate_several_variograms(ds_cmls, 'rainfall_ratio')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_mul_p_ok_vandebeek.csv', 
    'OpenRainER'
)        
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [01:39<00:00,  2.65it/s]


In [38]:
# multiplicative POINT ORDINARY KRIGING variogram

variograms = estimate_several_variograms(ds_cmls, 'rainfall_ratio')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_mul_p_ok_variogram.csv', 
    'OpenRainER'
)    
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [01:44<00:00,  2.54it/s]


In [39]:
# multiplicative POINT ORDINARY KRIGING variogram_fixnugget0

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_ratio', 0)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_mul_p_ok_variogram_fixnugget0.csv', 
    'OpenRainER'
)    
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [01:40<00:00,  2.64it/s]


In [40]:
# multiplicative POINT ORDINARY KRIGING variogram_fixnugget03

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_ratio', 0.3)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_mul_p_ok_variogram_fixnugget03.csv', 
    'OpenRainER'
)    
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [01:40<00:00,  2.63it/s]


In [41]:
# multiplicative POINT ORDINARY KRIGING variogram_fixnugget06

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_ratio', 0.6)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_mul_p_ok_variogram_fixnugget06.csv', 
    'OpenRainER'
)       
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [01:37<00:00,  2.70it/s]


In [42]:
# additive BLOCK ORDINARY KRIGING vandebeek

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_add_b_ok_vandebeek.csv', 
    'OpenRainER'
)   
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:42<00:00,  1.63it/s]


In [43]:
# additive BLOCK ORDINARY KRIGING variogram

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_add_b_ok_variogram.csv', 
    'OpenRainER'
)  
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:39<00:00,  1.65it/s]


In [44]:
# additive BLOCK ORDINARY KRIGING variogram_fixnugget0

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_add_b_ok_variogram_fixnugget0.csv', 
    'OpenRainER'
)       
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:36<00:00,  1.68it/s]


In [45]:
# additive BLOCK ORDINARY KRIGING variogram_fixnugget03

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.3)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_add_b_ok_variogram_fixnugget03.csv', 
    'OpenRainER'
)       
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:37<00:00,  1.68it/s]


In [46]:
# additive BLOCK ORDINARY KRIGING variogram_fixnugget06

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.6)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_add_b_ok_variogram_fixnugget06.csv', 
    'OpenRainER'
)       
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:59<00:00,  1.47it/s]


In [47]:
# multiplicative BLOCK ORDINARY KRIGING vandebeek

variograms = estimate_several_variograms(ds_cmls, 'rainfall_ratio')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_mul_b_ok_vandebeek.csv', 
    'OpenRainER'
)        
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [01:40<00:00,  2.62it/s]


In [48]:
# multiplicative BLOCK ORDINARY KRIGING variogram

variograms = estimate_several_variograms(ds_cmls, 'rainfall_ratio')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_mul_b_ok_variogram.csv', 
    'OpenRainER'
)   
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [01:35<00:00,  2.76it/s]


In [49]:
# multiplicative BLOCK ORDINARY KRIGING variogram_fixnugget0

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_ratio', 0)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_mul_b_ok_variogram_fixnugget0.csv', 
    'OpenRainER'
)     
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [01:39<00:00,  2.66it/s]


In [50]:
# multiplicative BLOCK ORDINARY KRIGING variogram_fixnugget03

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_ratio', 0.3)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_mul_b_ok_variogram_fixnugget03.csv', 
    'OpenRainER'
)       
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [01:37<00:00,  2.72it/s]


In [51]:
# multiplicative BLOCK ORDINARY KRIGING variogram_fixnugget06

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_ratio', 0.6)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
        log_transform=True,
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_mul_b_ok_variogram_fixnugget06.csv', 
    'OpenRainER'
)      
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [01:38<00:00,  2.67it/s]


In [52]:
# KED point vandebeek

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_ked_p_vandebeek.csv', 
    'OpenRainER'
)     
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:43<00:00,  1.62it/s]


In [53]:
# KED point variogram

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_ked_p_variogram.csv', 
    'OpenRainER'
)   
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:42<00:00,  1.63it/s]


In [54]:
# KED point variogram_fixnugget0

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_ked_p_variogram_fixnugget0.csv', 
    'OpenRainER'
)     
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:44<00:00,  1.61it/s]


In [55]:
# KED point variogram_fixnugget03

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.3)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_ked_p_variogram_fixnugget03.csv', 
    'OpenRainER'
)       
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:45<00:00,  1.60it/s]


In [56]:
# KED point variogram_fixnugget06

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.6)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_ked_p_variogram_fixnugget06.csv', 
    'OpenRainER'
)    
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:48<00:00,  1.56it/s]


In [57]:
# KED block vandebeek

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters={'nugget':0.3, 'psill':0.7, 'range': 30000},
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_ked_b_vandebeek.csv', 
    'OpenRainER'
)    
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:43<00:00,  1.62it/s]


In [58]:
# KED block variogram

variograms = estimate_several_variograms(ds_cmls, 'rainfall_difference')

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_ked_b_variogram.csv', 
    'OpenRainER'
)  
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:45<00:00,  1.60it/s]


In [59]:
# KED block variogram_fixnugget0

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_ked_b_variogram_fixnugget0.csv', 
    'OpenRainER'
)     
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:44<00:00,  1.60it/s]


In [60]:
# KED block variogram_fixnugget03

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.3)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_ked_b_variogram_fixnugget03.csv', 
    'OpenRainER'
)  
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:43<00:00,  1.62it/s]


In [61]:
# KED block variogram_fixnugget06

variograms = estimate_several_variograms_fixnugget(ds_cmls, 'rainfall_difference', 0.6)

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variograms.loc[time_start].to_dict(),
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

compute_metrics(
    data,
    'data/metrics/OpenRainER_ked_b_variogram_fixnugget06.csv', 
    'OpenRainER'
)   
del data, merger

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:44<00:00,  1.60it/s]
